In [7]:
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as transforms
from torchvision import datasets
import torch
import torch.nn as nn
import torch.optim as optim

# ===============================
# TRANSFORM
# ===============================
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

# ===============================
# DATASET LOAD
# ===============================
dataset = datasets.ImageFolder(
    root="dataset",   
    transform=transform
)

# ===============================
# SPLIT (IMPORTANT 🔥)
# ===============================
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

trainset, testset = random_split(dataset, [train_size, test_size])

# ===============================
# LOADERS
# ===============================
train_loader = DataLoader(trainset, batch_size=32, shuffle=True)
test_loader = DataLoader(testset, batch_size=32, shuffle=False)

# ===============================
# SIMPLE CNN MODEL
# ===============================
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 32 * 32, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

model = CNN()

# ===============================
# LOSS + OPTIMIZER
# ===============================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ===============================
# TRAIN
# ===============================
epochs = 3

for epoch in range(epochs):
    correct = 0

    for images, labels in train_loader:
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()

    print(f"Epoch {epoch+1} Accuracy:", correct/len(trainset))

# ===============================
# TEST
# ===============================
correct = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()

print("✅ Test Accuracy:", correct/len(testset))

C:\ProgramData\anaconda3\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 1 Accuracy: 0.6376637663766377
Epoch 2 Accuracy: 0.753075307530753
Epoch 3 Accuracy: 0.8064806480648065
✅ Test Accuracy: 0.771
